# CSIRO Image2Biomass - V9: Ensemble (V7+V8) with Aggressive TTA

Combines V7 (DINOv2) and V8 (DINOv2+Depth) with 16-transform TTA:
- **V7**: DINOv2 ViT-Base (LB: 0.58)
- **V8**: DINOv2 + Depth Anything v2 (LB: 0.62)
- **TTA**: 16 transforms (4 flips x 4 rotations)
- **Ensemble**: Weighted average (V8 weighted higher)

## Setup
1. Add both model datasets + competition data
2. **Set Internet to OFF**
3. Run all cells

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from transformers import AutoModelForDepthEstimation
import gc

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

In [ ]:
# Paths
TEST_CSV = '/kaggle/input/csiro-biomass/test.csv'
TEST_IMG_DIR = '/kaggle/input/csiro-biomass/test'
TRAIN_CSV = '/kaggle/input/csiro-biomass/train.csv'

# Model paths - adjust based on your dataset names
V7_MODEL_BASE = '/kaggle/input/image2biomass-dinov2-foundation/pytorch/default/1/dinov2_foundation_model'
V8_MODEL_BASE = '/kaggle/input/image2biomass-dinov2-depth/pytorch/default/1/dinov2_depth_model'
DEPTH_MODEL_PATH = f'{V8_MODEL_BASE}/depth_anything_v2_small'

N_FOLDS = 5
TARGET_NAMES = ['Dry_Clover_g', 'Dry_Dead_g', 'Dry_Green_g', 'Dry_Total_g', 'GDM_g']

# Ensemble weights (V8 performed better)
V7_WEIGHT = 0.3
V8_WEIGHT = 0.7

CONFIG_V7 = {
    'backbone': 'vit_base_patch14_dinov2',
    'image_size': 518,
    'features': 768,
    'dropout': 0.3
}

CONFIG_V8 = {
    'backbone': 'vit_base_patch14_dinov2',
    'image_size': 518,
    'features': 768,
    'dropout': 0.3
}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
print(f"Ensemble weights: V7={V7_WEIGHT}, V8={V8_WEIGHT}")

## Aggressive TTA Transforms (16 transforms)

In [ ]:
def get_tta_transforms(image_size):
    """Generate 16 TTA transforms: 4 flips x 4 rotations."""
    transforms = []
    
    # Base normalization
    normalize = A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    
    # 4 flip combinations
    flips = [
        [],  # No flip
        [A.HorizontalFlip(p=1.0)],
        [A.VerticalFlip(p=1.0)],
        [A.HorizontalFlip(p=1.0), A.VerticalFlip(p=1.0)],
    ]
    
    # 4 rotation angles
    rotations = [0, 90, 180, 270]
    
    for flip_transforms in flips:
        for angle in rotations:
            aug_list = [
                A.Resize(image_size, image_size),
            ]
            aug_list.extend(flip_transforms)
            if angle > 0:
                aug_list.append(A.Rotate(limit=(angle, angle), p=1.0, border_mode=0))
            aug_list.extend([normalize, ToTensorV2()])
            
            transforms.append(A.Compose(aug_list))
    
    return transforms

print(f"TTA transforms: {len(get_tta_transforms(518))}")

## Model Definitions

In [ ]:
class DINOv2Model(nn.Module):
    """V7: DINOv2 only model."""
    
    def __init__(self, backbone_name, num_features=768, dropout=0.3):
        super().__init__()
        self.target_names = TARGET_NAMES
        
        self.backbone = timm.create_model(
            backbone_name,
            pretrained=False,
            num_classes=0,
        )
        
        self.heads = nn.ModuleDict()
        for name in self.target_names:
            self.heads[name] = nn.Sequential(
                nn.Linear(num_features, 256),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(256, 64),
                nn.GELU(),
                nn.Linear(64, 1)
            )
    
    def forward(self, images):
        features = self.backbone(images)
        return {name: self.heads[name](features).squeeze(-1) for name in self.target_names}


class SharedDepthEstimator(nn.Module):
    """Depth estimator for V8."""
    
    def __init__(self, model_path):
        super().__init__()
        self.model = AutoModelForDepthEstimation.from_pretrained(model_path)
        self.model.eval()
        for param in self.model.parameters():
            param.requires_grad = False
    
    @torch.no_grad()
    def forward(self, images):
        B, C, H, W = images.shape
        outputs = self.model(images)
        depth = outputs.predicted_depth
        depth = F.interpolate(depth.unsqueeze(1), size=(H, W), mode='bilinear', align_corners=False)
        
        depth_flat = depth.view(B, -1)
        depth_min = depth_flat.min(dim=1, keepdim=True)[0].view(B, 1, 1, 1)
        depth_max = depth_flat.max(dim=1, keepdim=True)[0].view(B, 1, 1, 1)
        depth = (depth - depth_min) / (depth_max - depth_min + 1e-8)
        return depth


class DINOv2DepthModel(nn.Module):
    """V8: DINOv2 + Depth fusion model."""
    
    def __init__(self, backbone_name, num_features=768, dropout=0.3):
        super().__init__()
        self.target_names = TARGET_NAMES
        
        self.backbone = timm.create_model(
            backbone_name,
            pretrained=False,
            num_classes=0,
        )
        
        self.depth_encoder = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm2d(32),
            nn.GELU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.GELU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.GELU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        fused_features = num_features + 256
        
        self.heads = nn.ModuleDict()
        for name in self.target_names:
            self.heads[name] = nn.Sequential(
                nn.Linear(fused_features, 256),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(256, 64),
                nn.GELU(),
                nn.Linear(64, 1)
            )
    
    def forward(self, images, depth_maps):
        rgb_features = self.backbone(images)
        depth_features = self.depth_encoder(depth_maps)
        fused = torch.cat([rgb_features, depth_features], dim=1)
        return {name: self.heads[name](fused).squeeze(-1) for name in self.target_names}

print("Models defined")

## Dataset

In [ ]:
class BiomassTestDataset(Dataset):
    def __init__(self, csv_path, img_dir):
        self.img_dir = Path(img_dir)
        self.df = pd.read_csv(csv_path)
        self.df['image_id'] = self.df['sample_id'].str.split('__').str[0]
        self.image_ids = self.df['image_id'].unique()
        self.image_paths = self.df.groupby('image_id')['image_path'].first().to_dict()
    
    def __len__(self):
        return len(self.image_ids)
    
    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image_path = self.img_dir / Path(self.image_paths[image_id]).name
        image = np.array(Image.open(image_path).convert('RGB'))
        return {'image': image, 'image_id': image_id}

print("Dataset defined")

## Get Target Statistics

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
train_df['image_id'] = train_df['sample_id'].str.split('__').str[0]
train_wide = train_df.pivot_table(index='image_id', columns='target_name', values='target', aggfunc='first')

target_stats = {}
for name in TARGET_NAMES:
    values = train_wide[name].values
    target_stats[name] = {'mean': float(np.mean(values)), 'std': float(np.std(values)) + 1e-8}

print("Target stats loaded")

## Load Data and Compute Depth Maps

In [ ]:
# Load depth model for V8
print("Loading depth model...")
depth_model = SharedDepthEstimator(DEPTH_MODEL_PATH).to(DEVICE)

# Load dataset
dataset = BiomassTestDataset(TEST_CSV, TEST_IMG_DIR)

# Pre-compute depth maps and cache images
print("Pre-computing depth maps...")
base_transform = A.Compose([
    A.Resize(CONFIG_V8['image_size'], CONFIG_V8['image_size']),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

raw_images = {}  # Store raw images for TTA
depth_maps_cache = {}

with torch.no_grad():
    for i in tqdm(range(len(dataset)), desc="Computing depth"):
        sample = dataset[i]
        img_id = sample['image_id']
        raw_img = sample['image']
        
        raw_images[img_id] = raw_img
        
        # Compute depth
        img_tensor = base_transform(image=raw_img)['image'].unsqueeze(0).to(DEVICE)
        depth = depth_model(img_tensor)
        depth_maps_cache[img_id] = depth.cpu()

# Free depth model
del depth_model
torch.cuda.empty_cache()
gc.collect()

print(f"Cached {len(raw_images)} images and depth maps")

## Run V7 Inference with TTA

In [ ]:
def denormalize(pred_dict, stats):
    return {name: (val * stats[name]['std']) + stats[name]['mean'] for name, val in pred_dict.items()}

tta_transforms = get_tta_transforms(CONFIG_V7['image_size'])
image_ids_order = list(raw_images.keys())

print(f"Running V7 with {len(tta_transforms)}-transform TTA...")
v7_predictions = {img_id: {name: [] for name in TARGET_NAMES} for img_id in image_ids_order}

for fold_idx in range(N_FOLDS):
    print(f"\nV7 Fold {fold_idx + 1}/{N_FOLDS}...")
    
    model = DINOv2Model(
        backbone_name=CONFIG_V7['backbone'],
        num_features=CONFIG_V7['features'],
        dropout=CONFIG_V7['dropout']
    ).to(DEVICE)
    
    checkpoint_path = Path(V7_MODEL_BASE) / f'fold_{fold_idx}' / 'best_model.pth'
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    
    state_dict = checkpoint['model_state_dict']
    model_state = model.state_dict()
    for key in model_state.keys():
        if key in state_dict:
            model_state[key] = state_dict[key]
    model.load_state_dict(model_state)
    model.eval()
    
    with torch.no_grad():
        for img_id in tqdm(image_ids_order, desc=f"V7 Fold {fold_idx + 1}"):
            raw_img = raw_images[img_id]
            
            # TTA predictions
            for tta_transform in tta_transforms:
                img_tensor = tta_transform(image=raw_img)['image'].unsqueeze(0).to(DEVICE)
                pred = model(img_tensor)
                pred_dict = {name: pred[name][0].item() for name in TARGET_NAMES}
                pred_denorm = denormalize(pred_dict, target_stats)
                
                for name in TARGET_NAMES:
                    v7_predictions[img_id][name].append(pred_denorm[name])
    
    del model
    torch.cuda.empty_cache()

# Average V7 predictions
v7_avg = {}
for img_id in image_ids_order:
    v7_avg[img_id] = {name: np.mean(v7_predictions[img_id][name]) for name in TARGET_NAMES}

print(f"\nV7 predictions complete")

## Run V8 Inference with TTA

In [ ]:
print(f"Running V8 with {len(tta_transforms)}-transform TTA...")
v8_predictions = {img_id: {name: [] for name in TARGET_NAMES} for img_id in image_ids_order}

for fold_idx in range(N_FOLDS):
    print(f"\nV8 Fold {fold_idx + 1}/{N_FOLDS}...")
    
    model = DINOv2DepthModel(
        backbone_name=CONFIG_V8['backbone'],
        num_features=CONFIG_V8['features'],
        dropout=CONFIG_V8['dropout']
    ).to(DEVICE)
    
    checkpoint_path = Path(V8_MODEL_BASE) / f'fold_{fold_idx}' / 'best_model.pth'
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    
    state_dict = checkpoint['model_state_dict']
    model_state = model.state_dict()
    for key in model_state.keys():
        if key in state_dict:
            model_state[key] = state_dict[key]
    model.load_state_dict(model_state)
    model.eval()
    
    with torch.no_grad():
        for img_id in tqdm(image_ids_order, desc=f"V8 Fold {fold_idx + 1}"):
            raw_img = raw_images[img_id]
            depth_map = depth_maps_cache[img_id].to(DEVICE)
            
            # TTA predictions (apply same transforms to depth)
            for tta_transform in tta_transforms:
                img_tensor = tta_transform(image=raw_img)['image'].unsqueeze(0).to(DEVICE)
                pred = model(img_tensor, depth_map)
                pred_dict = {name: pred[name][0].item() for name in TARGET_NAMES}
                pred_denorm = denormalize(pred_dict, target_stats)
                
                for name in TARGET_NAMES:
                    v8_predictions[img_id][name].append(pred_denorm[name])
    
    del model
    torch.cuda.empty_cache()

# Average V8 predictions
v8_avg = {}
for img_id in image_ids_order:
    v8_avg[img_id] = {name: np.mean(v8_predictions[img_id][name]) for name in TARGET_NAMES}

print(f"\nV8 predictions complete")

## Ensemble and Apply Constraints

In [ ]:
# Weighted ensemble
print(f"Ensembling V7 (weight={V7_WEIGHT}) + V8 (weight={V8_WEIGHT})...")
ensemble_predictions = {}

for img_id in image_ids_order:
    ensemble_pred = {}
    for name in TARGET_NAMES:
        ensemble_pred[name] = V7_WEIGHT * v7_avg[img_id][name] + V8_WEIGHT * v8_avg[img_id][name]
    ensemble_predictions[img_id] = ensemble_pred

# Apply biological constraints
print("Applying constraints...")
for img_id in image_ids_order:
    pred = ensemble_predictions[img_id]
    
    # Clip negatives
    for name in TARGET_NAMES:
        pred[name] = max(0.0, pred[name])
    
    clover = pred['Dry_Clover_g']
    dead = pred['Dry_Dead_g']
    green = pred['Dry_Green_g']
    total = pred['Dry_Total_g']
    
    # Enforce: Total = Clover + Dead + Green
    component_sum = clover + dead + green
    new_total = (total + component_sum) / 2
    
    if component_sum > 0:
        scale = new_total / component_sum
        pred['Dry_Clover_g'] = clover * scale
        pred['Dry_Dead_g'] = dead * scale
        pred['Dry_Green_g'] = green * scale
    pred['Dry_Total_g'] = new_total
    
    ensemble_predictions[img_id] = pred

print("\nPredictions summary:")
for name in TARGET_NAMES:
    vals = [ensemble_predictions[img_id][name] for img_id in image_ids_order]
    print(f"  {name:<15} mean: {np.mean(vals):>8.2f}  min: {np.min(vals):>8.2f}  max: {np.max(vals):>8.2f}")

## Create Submission

In [ ]:
test_df = pd.read_csv(TEST_CSV)

submission_rows = []
for _, row in test_df.iterrows():
    sample_id = row['sample_id']
    image_id = sample_id.split('__')[0]
    target_name = row['target_name']
    
    pred_value = ensemble_predictions[image_id][target_name]
    
    submission_rows.append({
        'sample_id': sample_id,
        'target': pred_value
    })

submission_df = pd.DataFrame(submission_rows)
submission_df.to_csv('submission.csv', index=False)

print("Submission created!")
print(f"Shape: {submission_df.shape}")
print(submission_df.head(10))

In [ ]:
print("\n" + "="*70)
print("V9: Ensemble (V7 + V8) with 16-transform TTA")
print(f"V7 weight: {V7_WEIGHT}, V8 weight: {V8_WEIGHT}")
print(f"TTA transforms: {len(tta_transforms)}")
print(f"Total predictions per image: {N_FOLDS} folds x {len(tta_transforms)} TTA x 2 models")
print("Submission file ready: submission.csv")
print("="*70)